# CO₂–N₂O slab: temperature-dependent transmission (HITRAN · Voigt · Beer–Lambert)

## 1. Theory in brief

**Gas.** Ideal gas: $n = P/(k_BT)$, partial pressure $p_i = x_iP$, $n_i = x_i n$. At fixed $P$, cooling *raises* $n$.

**Line strength.** A line has position $\tilde\nu_0$ (cm$^{-1}$), lower-state energy $E''$ (cm$^{-1}$) and strength
$S$ in cm$^{-1}$/(molecule cm$^{-2}$) $=$ cm/molecule, with $\int\sigma\,d\tilde\nu=S$. HITRAN's $S$ already contains the natural isotopic
abundance, so $n$ is the *total* number density of the gas.

**Temperature scaling of $S$ (HITRAN definition):**
$$S(T)=S(T_{ref})\,\frac{Q(T_{ref})}{Q(T)}\;\frac{e^{-c_2E''/T}}{e^{-c_2E''/T_{ref}}}\;\frac{1-e^{-c_2\tilde\nu_0/T}}{1-e^{-c_2\tilde\nu_0/T_{ref}}},\qquad T_{ref}=296\,\mathrm K,\; c_2=\tfrac{hc}{k_B}=1.4388\ \mathrm{cm\,K}$$
* $Q(T)$ (partition function) grows with $T$ as more states are populated → lowers *every* line.
* $e^{-c_2E''/T}$ moves population to high-$E''$ levels → raises high-$E''$ lines, lowers low-$E''$ lines.
* The last factor is the stimulated-emission correction ($\approx1$ here since $c_2\tilde\nu_0/T\approx13$).
* There is **no universal trend**. With $Q\propto T$ a line's strength peaks at $T=c_2E''$. For an optically thin line at *fixed pressure*, $W\propto nS$, so
  $d\ln W/d\ln T=-2+c_2E''/T$: only lines with $E''>2T/c_2$ strengthen on warming.

**Widths (always HWHM, never FWHM).**
$$\gamma_D=\frac{\tilde\nu_0}{c}\sqrt{\frac{2k_BT\ln2}{m}},\qquad
\gamma_L=\Big(\tfrac{T_{ref}}{T}\Big)^{n_{air}}\big[\gamma_{air}(P-p_{self})+\gamma_{self}\,p_{self}\big]\ (\text{pressures in atm}),\qquad \tilde\nu_0'=\tilde\nu_0+\delta_{air}P/\mathrm{atm}$$
Doppler width comes from the Maxwell velocity distribution (Gaussian); collisional width from collision-shortened state lifetimes (Lorentzian).
They are independent, so the line shape is their convolution, the Voigt profile
$$\varphi(\Delta\tilde\nu)=\frac{\mathrm{Re}\,w(z)}{\sigma_G\sqrt{2\pi}},\quad z=\frac{\Delta\tilde\nu+i\gamma_L}{\sigma_G\sqrt2},\quad \sigma_G=\frac{\gamma_D}{\sqrt{2\ln2}}$$
with $w$ the Faddeeva function. $\int\varphi\,d\tilde\nu=1$ exactly; a finite window misses a Lorentzian tail $\approx 2\gamma_L/(\pi W)$.

**Absorption and transmission.**
$$\sigma_i=\sum_k S_k(T)\varphi_k,\quad \alpha_i=n_i\sigma_i,\quad \alpha=\alpha_{CO_2}+\alpha_{N_2O},\quad \tau=\alpha L,\quad \mathcal T=e^{-\tau},\quad I=I_0\mathcal T$$
Intensity $I$ and transmittance $\mathcal T$ are different quantities; with a flat $I_0$ they have the same shape, with a blackbody $I_0$ they do not.
Diagnostics: $\int\tau\,d\tilde\nu$ and the equivalent width $W=\int(1-\mathcal T)\,d\tilde\nu$ (cm$^{-1}$). Neither is "absorbed energy".

**Thin vs saturated lines.** Thin: $W\approx SN$ ($N=nL$). Strongly saturated Lorentzian line: $W\approx2\sqrt{SN\gamma_L}$ (only wings still absorb).

**Where it fails.** Resolution effects at large $\tau$; CO₂ far wings under the N₂O band (a truncated Voigt ignores sub-Lorentzian wings and line mixing);
hot gas (HITRAN lacks weak hot-band lines; use HITEMP); large mole fractions (HITRAN has no CO₂/N₂O cross-broadening, air is used instead);
no emission, scattering, water vapour, or continuum.

## 2. Setup and settings

In [4]:
import os, sys, json, glob, re, time, datetime
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import wofz

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    print("ipywidgets", widgets.__version__)
except ImportError:
    print("ipywidgets missing: run  %pip install -U ipywidgets  and restart the kernel (the static draw() fallback still works)")
try:
    import hapi
    print("HAPI", getattr(hapi, "__version__", "?"))
except ImportError:
    print("HAPI missing: run  %pip install hitran-api  (needed for partition sums and downloading)")
print("numpy", np.__version__)

ipywidgets 8.1.9
HAPI version: 1.3.0.0
To get the most up-to-date version please check http://hitran.org/hapi
ATTENTION: Python versions of partition sums from TIPS-2021 are now available in HAPI code

           MIT license: Copyright 2021 HITRAN team, see more at http://hitran.org. 

           If you use HAPI in your research or software development,
           please cite it using the following reference:
           R.V. Kochanov, I.E. Gordon, L.S. Rothman, P. Wcislo, C. Hill, J.S. Wilzewski,
           HITRAN Application Programming Interface (HAPI): A comprehensive approach
           to working with spectroscopic data, J. Quant. Spectrosc. Radiat. Transfer 177, 15-30 (2016)
           DOI: 10.1016/j.jqsrt.2016.03.005

           ATTENTION: This is the core version of the HITRAN Application Programming Interface.
                      For more efficient implementation of the absorption coefficient routine, 
                      as well as for new profiles, parameters and other f

In [5]:
class cfg:
    # ---- spectral interval
    NU_MIN, NU_MAX = 2180.0, 2400.0     # cm^-1  (4.587 - 4.167 um; lambda[um] = 1e4/nu)
    NU_STEP   = 0.005                   # cm^-1; must be well below the Lorentz HWHM (~0.04 cm^-1)
    WING_CM1  = 10.0                    # each line evaluated within +/- this (model choice)
    S_MIN     = 1.0e-26                 # drop lines weaker than this at 296 K [cm/molecule] (model choice)
    # ---- atmospheric layer: US Standard Atmosphere 1976, troposphere ----
    LAYER_H_M = 5000.0                  # 5 km -> T = 255.65 K, P = 54 020 Pa
    T_SWEEP   = (216.65, 288.15)        # K: tropopause ... sea level
    N_FRAMES  = 13
    # ---- composition (dry-air mole fractions, uniform) ----
    X_CO2 = 423.03e-6                   # NOAA GML global monthly mean, Sep 2025
    X_N2O = 339.76e-9                   # NOAA GML global monthly mean, Mar 2026
    # ---- slab and incident radiation ----
    PATH_LENGTH_M = 100.0               # initial guess; adjust with the slider
    T_SOURCE_K    = 1500.0              # blackbody source (illustrative lamp)
    # ---- data ----
    DATA_DIR = "data"
    FETCH_MARGIN_CM1 = 25.0

## 3. Physics engine

Units: $\tilde\nu$ cm$^{-1}$; $S$ cm/molecule; $\varphi$ cm; $\sigma$ cm$^2$/molecule; $n$ cm$^{-3}$; $\alpha$ cm$^{-1}$; $L$ cm; widths are HWHM.

In [7]:
"""
rt_core.py - physics engine for the CO2 + N2O transmission model.


UNIT CONVENTIONS (HITRAN-style, CGS spectroscopy units)
  wavenumber nu ............ cm^-1
  line intensity S ......... cm^-1 / (molecule cm^-2)  == cm / molecule
  line-shape phi(nu) ....... cm            (integrates to 1 over cm^-1)
  cross section sigma ...... cm^2 / molecule   = S * phi
  number density n ......... molecule / cm^3
  absorption coeff alpha ... cm^-1         = n * sigma
  path length L ............ cm
  optical depth tau ........ dimensionless = alpha * L
  HWHM gamma ............... cm^-1  (HALF width at HALF maximum - NOT FWHM,
                                     NOT the 1/e half-width)
"""


# NumPy 2.x renamed trapz -> trapezoid; support both.
trapezoid = getattr(np, "trapezoid", None) or np.trapz

# ----------------------------------------------------------------------------
# Constants (SI values are exact since the 2019 SI redefinition)
# ----------------------------------------------------------------------------
H_PLANCK = 6.62607015e-34      # J s
C_LIGHT = 299792458.0          # m / s
K_B = 1.380649e-23             # J / K
AMU = 1.66053906660e-27        # kg (atomic mass unit; CODATA 2018 value)
ATM_PA = 101325.0              # Pa per standard atmosphere
T_REF = 296.0                  # K, HITRAN reference temperature
# second radiation constant c2 = h c / k_B, converted from m K to cm K
C2_CM_K = H_PLANCK * C_LIGHT / K_B * 100.0
LN2 = np.log(2.0)


# ----------------------------------------------------------------------------
# Data container for one gas
# ----------------------------------------------------------------------------
@dataclass
class LineList:
    """HITRAN line parameters for one molecule (all isotopologues).

    S is the HITRAN intensity at T_REF = 296 K. HITRAN's `sw` column already
    includes the natural isotopic abundance, so n_species below must be the
    TOTAL molecule number density (do not multiply by abundance again).
    """
    name: str
    M: int                      # HITRAN molecule id (CO2 = 2, N2O = 4)
    nu: np.ndarray              # cm^-1
    S: np.ndarray               # cm^-1/(molecule cm^-2) at 296 K
    E_low: np.ndarray           # cm^-1, lower-state energy
    gamma_air: np.ndarray       # cm^-1/atm, HWHM at 296 K, 1 atm
    gamma_self: np.ndarray      # cm^-1/atm, HWHM at 296 K, 1 atm
    n_air: np.ndarray           # temperature exponent of gamma_air
    delta_air: np.ndarray       # cm^-1/atm, pressure shift
    iso: np.ndarray             # local isotopologue index I (int)
    mass_amu: dict              # I -> molecular mass in amu

    def select(self, mask):
        return LineList(self.name, self.M, self.nu[mask], self.S[mask],
                        self.E_low[mask], self.gamma_air[mask],
                        self.gamma_self[mask], self.n_air[mask],
                        self.delta_air[mask], self.iso[mask], self.mass_amu)


# ----------------------------------------------------------------------------
# Ideal-gas mixture
# ----------------------------------------------------------------------------
def number_density_cm3(T, P_pa):
    """Total molecule number density n = P / (k_B T), returned in cm^-3.

    Assumes an ideal gas (valid here: ~1e5 Pa or less, 200-300 K, where the
    second virial correction is small). Partial pressure of species i is
    p_i = x_i P (Dalton) and n_i = x_i n for an ideal-gas mixture.
    """
    return P_pa / (K_B * T) * 1.0e-6        # m^-3 -> cm^-3


def dilution_warnings(x_co2, x_n2o, limit=0.01):
    """Air-broadening is used for the 'foreign' gas, which is only defensible
    if both absorbers are trace gases. Return warning strings otherwise."""
    if x_co2 < 0 or x_n2o < 0 or x_co2 + x_n2o > 1:
        raise ValueError("mole fractions must be >= 0 and sum to <= 1")
    msgs = []
    if x_co2 > limit or x_n2o > limit:
        msgs.append(
            "WARNING: x_CO2 or x_N2O exceeds %.0f%%. The model uses HITRAN "
            "air-broadening for all non-self collisions; CO2- and N2O-"
            "broadening of each other's lines is NOT available in the "
            "standard HITRAN line list, so results are an approximation "
            "and not validated in this regime." % (limit * 100))
    return msgs


# ----------------------------------------------------------------------------
# Temperature dependence of the line intensity (HITRAN definition)
#   S(T) = S(Tref) * [Q(Tref)/Q(T)]
#                  * exp(-c2 E''/T) / exp(-c2 E''/Tref)
#                  * [1 - exp(-c2 nu/T)] / [1 - exp(-c2 nu/Tref)]
# ----------------------------------------------------------------------------
def scale_intensity(S_ref, nu0, E_low, Q_ref_over_Q, T):
    """Scale HITRAN intensities from T_REF to T.

    Q_ref_over_Q : Q(T_REF)/Q(T), the partition-function ratio (per line)
    rB           : ratio of Boltzmann populations of the lower state
    rE           : ratio of stimulated-emission factors
    """
    rB = np.exp(-C2_CM_K * E_low * (1.0 / T - 1.0 / T_REF))
    rE = (-np.expm1(-C2_CM_K * nu0 / T)) / (-np.expm1(-C2_CM_K * nu0 / T_REF))
    return S_ref * Q_ref_over_Q * rB * rE


def partition_ratio_per_line(lines, T, partition_fn):
    """Q(T_REF)/Q(T) for every line (depends on the line's isotopologue).
    partition_fn(M, I, T) -> total internal partition sum (dimensionless)."""
    ratio = np.empty(len(lines.nu))
    for I in np.unique(lines.iso):
        r = partition_fn(lines.M, int(I), T_REF) / partition_fn(lines.M, int(I), T)
        ratio[lines.iso == I] = r
    return ratio


# ----------------------------------------------------------------------------
# Line widths
# ----------------------------------------------------------------------------
def doppler_hwhm(nu0, T, mass_amu):
    """Doppler HWHM [cm^-1] = (nu0/c) * sqrt(2 k_B T ln2 / m).
    (HWHM. The 1/e half-width is larger by 1/sqrt(ln2); FWHM is 2x HWHM.)"""
    m = np.asarray(mass_amu) * AMU
    return nu0 / C_LIGHT * np.sqrt(2.0 * K_B * T * LN2 / m)


def lorentz_hwhm(gamma_air, gamma_self, n_air, T, P_pa, x_self):
    """Collisional (Lorentz) HWHM [cm^-1] following the HITRAN definition:
         gamma_L = (Tref/T)^n_air * [ gamma_air (P - p_self) + gamma_self p_self ]
    with pressures in atm. 'Foreign' collisions are treated as air.
    The single exponent n_air is applied to the self part too (HITRAN
    provides no separate exponent in the standard list)."""
    P_atm = P_pa / ATM_PA
    p_self = x_self * P_atm
    return (T_REF / T) ** n_air * (gamma_air * (P_atm - p_self) + gamma_self * p_self)


# ----------------------------------------------------------------------------
# Voigt profile (Doppler Gaussian convolved with collisional Lorentzian)
# ----------------------------------------------------------------------------
def voigt_profile(dnu, gamma_D, gamma_L):
    """Normalised Voigt profile phi(dnu) [cm], with dnu = nu - nu0 [cm^-1].

    gamma_D, gamma_L are HWHM values. Gaussian sigma = gamma_D / sqrt(2 ln2).
    phi = Re[w(z)] / (sigma sqrt(2 pi)),   z = (dnu + i gamma_L) / (sigma sqrt 2)
    where w is the Faddeeva function (scipy.special.wofz).
    Integral of phi over all dnu is exactly 1.
    """
    sigma = gamma_D / np.sqrt(2.0 * LN2)
    z = (dnu + 1j * gamma_L) / (sigma * np.sqrt(2.0))
    return wofz(z).real / (sigma * np.sqrt(2.0 * np.pi))


# ----------------------------------------------------------------------------
# Cross section and absorption coefficient
# ----------------------------------------------------------------------------
def species_cross_section(grid, lines, T, P_pa, x_self, partition_fn,
                          wing_cm1=10.0, s_min=0.0):
    """sigma(nu) [cm^2/molecule] = sum_i S_i(T) * phi_i(nu), natural abundance
    included via HITRAN's S. Each line is evaluated only within +/- wing_cm1
    of its (pressure-shifted) centre: this truncation is a MODEL CHOICE
    (it drops the far Lorentzian wings and ignores sub-Lorentzian behaviour).
    """
    P_atm = P_pa / ATM_PA
    keep = ((lines.S >= s_min) & (lines.nu > grid[0] - wing_cm1)
            & (lines.nu < grid[-1] + wing_cm1))
    L = lines.select(keep)
    sigma = np.zeros_like(grid)
    if len(L.nu) == 0:
        return sigma
    S_T = scale_intensity(L.S, L.nu, L.E_low,
                          partition_ratio_per_line(L, T, partition_fn), T)
    mass = np.array([L.mass_amu[int(i)] for i in L.iso])
    nu_c = L.nu + L.delta_air * P_atm             # HITRAN pressure shift
    gD = doppler_hwhm(nu_c, T, mass)
    gL = lorentz_hwhm(L.gamma_air, L.gamma_self, L.n_air, T, P_pa, x_self)
    lo = np.searchsorted(grid, nu_c - wing_cm1)
    hi = np.searchsorted(grid, nu_c + wing_cm1)
    for k in range(len(nu_c)):
        if hi[k] > lo[k]:
            sigma[lo[k]:hi[k]] += S_T[k] * voigt_profile(
                grid[lo[k]:hi[k]] - nu_c[k], gD[k], gL[k])
    return sigma


def absorption_coefficients(grid, co2_lines, n2o_lines, T, P_pa, x_co2, x_n2o,
                            partition_fn, wing_cm1=10.0, s_min=0.0):
    """Return (alpha_CO2, alpha_N2O) in cm^-1.
    alpha_i = n_i * sigma_i,  n_i = x_i * P/(k_B T).
    The two are additive: alpha_total = alpha_CO2 + alpha_N2O."""
    n_tot = number_density_cm3(T, P_pa)
    a_co2 = x_co2 * n_tot * species_cross_section(
        grid, co2_lines, T, P_pa, x_co2, partition_fn, wing_cm1, s_min)
    a_n2o = x_n2o * n_tot * species_cross_section(
        grid, n2o_lines, T, P_pa, x_n2o, partition_fn, wing_cm1, s_min)
    return a_co2, a_n2o


# ----------------------------------------------------------------------------
# Beer-Lambert transmission
# ----------------------------------------------------------------------------
def beer_lambert(alpha_total, L_cm, I0):
    """tau = alpha L ; transmittance = exp(-tau) ; I = I0 * transmittance.
    Valid for a uniform, non-scattering, non-emitting slab and radiation
    that is narrow compared with the line structure (monochromatic)."""
    tau = alpha_total * L_cm
    trans = np.exp(-tau)
    return tau, trans, I0 * trans


def integrated_tau(grid, alpha, L_cm):
    """Integral of tau(nu) d nu  [cm^-1]  (a spectrally integrated optical depth).
    NOT an absorbed energy: that would need I0(nu) weighting and saturation."""
    return trapezoid(alpha * L_cm, grid)


def equivalent_width(grid, trans):
    """W = integral of (1 - T(nu)) d nu  [cm^-1]; spectral width of a fully
    opaque band that would absorb the same amount of a FLAT incident spectrum."""
    return trapezoid(1.0 - trans, grid)


def line_strength_vs_T(lines, k, Ts, partition_fn):
    """S(T) [cm/molecule] for line index k at each temperature in Ts."""
    one = lines.select(np.arange(len(lines.nu)) == k)
    out = []
    for T in Ts:
        qr = partition_ratio_per_line(one, T, partition_fn)
        out.append(float(scale_intensity(one.S, one.nu, one.E_low, qr, T)[0]))
    return np.array(out)


# ----------------------------------------------------------------------------
# Incident spectrum
# ----------------------------------------------------------------------------
def planck_per_cm1(nu_cm1, T_K):
    """Planck spectral radiance per unit wavenumber, W m^-2 sr^-1 (cm^-1)^-1.
    B = 2 h c^2 nu^3 / (exp(h c nu / k T) - 1) with nu in m^-1, times
    100 m^-1 per cm^-1 to convert the per-wavenumber unit."""
    nu_m = np.asarray(nu_cm1) * 100.0
    x = H_PLANCK * C_LIGHT * nu_m / (K_B * T_K)
    return 2.0 * H_PLANCK * C_LIGHT ** 2 * nu_m ** 3 / np.expm1(x) * 100.0


# ----------------------------------------------------------------------------
# US Standard Atmosphere 1976, troposphere (0-11 km geopotential height)
# ----------------------------------------------------------------------------
def isa_troposphere(h_m):
    """Temperature [K] and pressure [Pa] in the ISA/US-1976 troposphere:
    T = 288.15 K - 0.0065 K/m * h ;  P = 101325 Pa * (T/288.15)^(g0 M0 / R* L)."""
    g0, M0, R_star, lapse = 9.80665, 0.0289644, 8.31432, 0.0065
    T = 288.15 - lapse * np.asarray(h_m, dtype=float)
    P = 101325.0 * (T / 288.15) ** (g0 * M0 / (R_star * lapse))
    return T, P

## 4. HITRAN input/output and downloading

Lines are read from HITRAN's fixed-width `.par` format (HAPI's `.data` files use the same format), so you can either download with HAPI
**or** download manually from hitran.org (Data Access → Line-by-line, CO₂ and N₂O, all isotopologues, about 2155–2425 cm⁻¹, `.par`) and save as
`data/CO2_2155_2425.par` and `data/N2O_2155_2425.par`. HAPI is always used for isotopologue masses and the partition sums.
This notebook contains **no API key**; how HAPI 1.3.x picks the key up is not something I could verify, so check HAPI's page
(https://hitran.org/hapi/) if the download fails, and use the manual route as the fallback.

In [9]:
"""
py - read HITRAN line lists and talk to HAPI.

* Lines are read by our own parser from HITRAN's fixed-width 160-character
  .par format, so they can come from HAPI's fetch (.data files in the same
  format) OR from a manual download on hitran.org (.par files).
* HAPI is used only for the isotopologue table (masses, abundances) and for
  the partition sums Q(T) (TIPS, computed locally, no internet needed).
"""



# 0-based slice positions of the fields we need in a 160-char .par line
_SLICES = dict(M=(0, 2), I=(2, 3), nu=(3, 15), S=(15, 25), A=(25, 35),
               gamma_air=(35, 40), gamma_self=(40, 45), E_low=(45, 55),
               n_air=(55, 59), delta_air=(59, 67))


def _to_float(txt):
    """Float parser tolerant of Fortran 'D' exponents and HITRAN's odd '2.700-164'."""
    t = txt.strip().replace("D", "E")
    try:
        return float(t)
    except ValueError:
        m = re.fullmatch(r"(\d\.\d{3})-(\d{3})", t)
        if m:
            return float(m.group(1) + "E-" + m.group(2))
        raise


def _iso_index(ch):
    """Local isotopologue id: '1'-'9', '0' -> 10, 'A' -> 11, 'B' -> 12 ..."""
    if ch == "0":
        return 10
    if ch.isdigit():
        return int(ch)
    return 11 + ord(ch) - ord("A")


def parse_par_files(paths, M_expected):
    """Parse .par / .data files into column arrays (dict)."""
    cols = {k: [] for k in _SLICES}
    for path in paths:
        with open(path, "r") as fh:
            for line in fh:
                if len(line.rstrip("\n")) < 67:
                    continue
                if int(line[0:2]) != M_expected:
                    continue
                for key, (a, b) in _SLICES.items():
                    seg = line[a:b]
                    cols[key].append(_iso_index(seg) if key == "I"
                                     else int(seg) if key == "M" else _to_float(seg))
    return {k: np.array(v) for k, v in cols.items()}


def repair_missing(cols):
    """Documented policy for unusable broadening parameters (<= 0 or NaN).

    HITRAN uses placeholder values where a parameter is not available. Rather
    than invent numbers, missing values are replaced by DATA-DERIVED estimates
    (median of valid lines of the same isotopologue, else of the same gas),
    and the number of replaced lines is returned so it is reported openly.
    gamma_self falls back to the (repaired) gamma_air; because x_self is
    ~1e-4 the effect on the width is of order x_self * |gamma_self - gamma_air|.
    """
    report = {}
    for key in ("gamma_air", "n_air"):
        bad = ~(cols[key] > 0)
        report[key] = int(bad.sum())
        for I in np.unique(cols["I"][bad]) if bad.any() else []:
            sel = cols["I"] == I
            good = sel & ~bad
            ref = (np.median(cols[key][good]) if good.any()
                   else np.median(cols[key][~bad]))
            cols[key][sel & bad] = ref
    bad = ~(cols["gamma_self"] > 0)
    report["gamma_self"] = int(bad.sum())
    cols["gamma_self"][bad] = cols["gamma_air"][bad]
    return report


def iso_table(M):
    """{local I: (global_id, name, abundance, mass_amu)} from HAPI's ISO table."""
    import hapi
    out = {}
    for (m, I), v in hapi.ISO.items():
        if m == M:
            out[I] = (v[hapi.ISO_INDEX["id"]], v[hapi.ISO_INDEX["iso_name"]],
                      v[hapi.ISO_INDEX["abundance"]], v[hapi.ISO_INDEX["mass"]])
    return out


def hapi_partition_fn(M, I, T):
    """Total internal partition sum Q(M, I, T) from HAPI's built-in TIPS.
    (Which TIPS edition is used depends on the installed HAPI version.)"""
    import hapi
    q = hapi.partitionSum(M, I, float(T))
    if isinstance(q, tuple):
        q = q[1]
    return float(np.atleast_1d(q)[0])


def load_linelist(name, M, data_dir="data"):
    """Load all HITRAN files for one gas. Looks for <name>*.data and <name>*.par."""
    paths = sorted(glob.glob(os.path.join(data_dir, name + "*.data"))
                   + glob.glob(os.path.join(data_dir, name + "*.par")))
    if not paths:
        raise FileNotFoundError(
            "No %s line list in '%s'. Run fetch_hitran.py, or download a .par "
            "file from hitran.org and name it e.g. %s_2155_2425.par."
            % (name, data_dir, name))
    cols = parse_par_files(paths, M)
    if len(cols["nu"]) == 0:
        raise ValueError("Files %s contain no lines for molecule id %d" % (paths, M))
    report = repair_missing(cols)
    masses = {I: v[3] for I, v in iso_table(M).items()}
    lines = LineList(name, M, cols["nu"], cols["S"], cols["E_low"],
                     cols["gamma_air"], cols["gamma_self"], cols["n_air"],
                     cols["delta_air"], cols["I"].astype(int), masses)
    return lines, report, paths

In [10]:
MOLECULES = (("CO2", 2), ("N2O", 4))      # HITRAN molecule ids

def fetch_hitran():
    # Download CO2 and N2O line lists with HAPI and write data/data_manifest.json.
    try:
        import hapi
    except ImportError:
        raise RuntimeError("HAPI not installed. Run: %pip install hitran-api")
    os.makedirs(cfg.DATA_DIR, exist_ok=True)
    hapi.db_begin(cfg.DATA_DIR)
    numin, numax = cfg.NU_MIN - cfg.FETCH_MARGIN_CM1, cfg.NU_MAX + cfg.FETCH_MARGIN_CM1
    hapi.VARIABLES["DISPLAY_FETCH_URL"] = True
    manifest = {"retrieval_date": datetime.date.today().isoformat(), "source": "HITRANonline via HAPI",
                "hapi_version": getattr(hapi, "__version__", "unknown"),
                "wavenumber_range_cm-1": [numin, numax],
                "units": {"nu": "cm-1", "S": "cm-1/(molecule cm-2) at 296 K, abundance-weighted",
                          "gamma": "cm-1/atm HWHM at 296 K", "E_low": "cm-1"}, "gases": {}}
    for name, M in MOLECULES:
        iso = iso_table(M)
        table = "%s_%d_%d" % (name, round(numin), round(numax))
        try:
            hapi.fetch_by_ids(table, [v[0] for v in iso.values()], numin, numax)
        except Exception as exc:
            raise RuntimeError("Download failed for %s: %s\nUse the manual .par route described above." % (name, exc))
        manifest["gases"][name] = {"hitran_molecule_id": M, "table": table,
            "isotopologues": {str(I): {"global_id": v[0], "name": v[1], "abundance": v[2], "mass_amu": v[3]}
                              for I, v in iso.items()}}
    for name, M in MOLECULES:
        lines, report, paths = load_linelist(name, M, cfg.DATA_DIR)
        manifest["gases"][name].update(n_lines=int(len(lines.nu)), files=[os.path.basename(p) for p in paths],
                                       parameters_replaced_by_data_derived_estimate=report)
        print("%s: %d lines; replaced parameters: %s" % (name, len(lines.nu), report))
    with open(os.path.join(cfg.DATA_DIR, "data_manifest.json"), "w") as fh:
        json.dump(manifest, fh, indent=2)
    print("Wrote", os.path.join(cfg.DATA_DIR, "data_manifest.json"))

## 5. Tests

These 19 tests use **synthetic** lines and a toy partition function; they test the equations and the code, not HITRAN and not the real spectrum.

In [12]:
def toy_Q(M, I, T):
    return T


def make_lines(nu, S, E=0.0, g_air=0.07, g_self=0.07, n=0.7, delta=0.0, M=2, mass=44.0):
    nu = np.atleast_1d(np.asarray(nu, float))
    k = len(nu)
    full = lambda v: np.full(k, v, float) if np.isscalar(v) else np.asarray(v, float)
    return LineList("toy", M, nu, full(S), full(E), full(g_air), full(g_self),
                       full(n), full(delta), np.ones(k, int), {1: mass})


# ---------------------------------------------------------------- line shape
def _norm_with_tail(gD, gL, W, step):
    x = np.arange(-W, W + step, step)
    area = trapezoid(voigt_profile(x, gD, gL), x)
    return area, area + 2 * gL / (np.pi * W)       # + analytic Lorentzian tail


def test_voigt_normalisation():
    # Raw integral over +/-W misses the Lorentzian tail ~ 2 gL/(pi W);
    # adding that tail analytically must give 1 to high accuracy.
    for gD, gL in [(0.002, 0.04), (0.01, 0.001), (0.01, 0.0), (0.002, 0.2)]:
        raw, corrected = _norm_with_tail(gD, gL, W=2000.0, step=min(gD, max(gL, 1e-3)) / 8)
        assert abs(corrected - 1.0) < 2e-5, (gD, gL, raw, corrected)
        assert raw <= 1.0 + 1e-9


def test_voigt_limits_and_hwhm_convention():
    # pure Gaussian: peak 1/(sigma sqrt(2 pi)); value at dnu = gD must be half the peak (HWHM!)
    gD = 0.01
    peak = voigt_profile(0.0, gD, 0.0)
    assert abs(voigt_profile(gD, gD, 0.0) / peak - 0.5) < 1e-12
    sigma = gD / np.sqrt(2 * np.log(2))
    assert abs(peak - 1 / (sigma * np.sqrt(2 * np.pi))) < 1e-9 * peak
    # pure Lorentzian: peak 1/(pi gL); half maximum at dnu = gL
    gL = 0.05
    p = voigt_profile(0.0, 1e-7, gL)
    assert abs(p - 1 / (np.pi * gL)) / p < 1e-4
    assert abs(voigt_profile(gL, 1e-7, gL) / p - 0.5) < 1e-4


def test_doppler_width_scaling_and_value():
    a = doppler_hwhm(2349.0, 250.0, 44.0)
    b = doppler_hwhm(2349.0, 1000.0, 44.0)
    assert abs(b / a - 2.0) < 1e-12                       # proportional to sqrt(T)
    ref = 2349.0 / C_LIGHT * np.sqrt(2 * 1.380649e-23 * 250 * np.log(2) / (44.0 * 1.66053906660e-27))
    assert abs(a - ref) / ref < 1e-12


# ------------------------------------------------------- intensity scaling
def test_intensity_scaling_identity_and_formula():
    S = scale_intensity(1e-20, 2300.0, 500.0, 1.0, T_REF)
    assert abs(S - 1e-20) < 1e-30                          # T = T_ref -> unchanged
    T, nu, E, S0 = 230.0, 2224.0, 300.0, 2e-19
    c2 = 1.438776877
    expected = (S0 * (T_REF / T) * np.exp(-c2 * E / T) / np.exp(-c2 * E / T_REF)
                * (1 - np.exp(-c2 * nu / T)) / (1 - np.exp(-c2 * nu / T_REF)))
    got = scale_intensity(S0, nu, E, toy_Q(0, 0, T_REF) / toy_Q(0, 0, T), T)
    assert abs(got - expected) / expected < 1e-8


def test_c2_value():
    assert abs(C2_CM_K - 1.438776877) < 1e-8


def test_temperature_trends_differ_between_transitions():
    # Low-E'' line weakens on heating, high-E'' line strengthens: no universal trend.
    low = [scale_intensity(1e-19, 2300.0, 0.0, T_REF / T, T) for T in (250, 296, 400)]
    high = [scale_intensity(1e-19, 2300.0, 1500.0, T_REF / T, T) for T in (250, 296, 400)]
    assert low[0] > low[1] > low[2]
    assert high[0] < high[1] < high[2]


def test_stimulated_emission_rayleigh_jeans_limit():
    rE = scale_intensity(1.0, 1e-4, 0.0, 1.0, 200.0)   # nu -> 0: ratio -> T_ref/T
    assert abs(rE - T_REF / 200.0) < 1e-6


# -------------------------------------------------------------- broadening
def test_pressure_and_temperature_dependence_of_lorentz_width():
    g = lorentz_hwhm(0.07, 0.09, 0.7, 296.0, 101325.0, 0.0)
    assert abs(g - 0.07) < 1e-12                           # 1 atm, 296 K, no self
    g = lorentz_hwhm(0.07, 0.09, 0.7, 296.0, 0.5 * 101325.0, 0.0)
    assert abs(g - 0.035) < 1e-12                          # linear in P
    g = lorentz_hwhm(0.07, 0.09, 0.7, 200.0, 101325.0, 0.0)
    assert abs(g - 0.07 * (296 / 200) ** 0.7) < 1e-12
    g = lorentz_hwhm(0.07, 0.09, 0.7, 296.0, 101325.0, 0.1)   # self part mixed in
    assert abs(g - (0.07 * 0.9 + 0.09 * 0.1)) < 1e-12


# ------------------------------------------------- units / sum rule / gas law
def test_loschmidt_number_density():
    n = number_density_cm3(273.15, 101325.0) * 1e6      # back to m^-3
    assert abs(n - 2.686780111e25) / 2.686780111e25 < 1e-6


def test_line_strength_sum_rule():
    # integral of sigma d nu over an isolated line must equal S(T) [cm/molecule]
    nu0, S0, W = 2300.0, 1e-18, 400.0
    grid = np.arange(nu0 - W, nu0 + W, 0.005)
    lines = make_lines([nu0], [S0], E=200.0)
    sig = species_cross_section(grid, lines, 250.0, 0.5 * 101325, 0.0, toy_Q, wing_cm1=W)
    S_T = scale_intensity(S0, nu0, 200.0, T_REF / 250.0, 250.0)
    gL = lorentz_hwhm(0.07, 0.07, 0.7, 250.0, 0.5 * 101325, 0.0)
    area = trapezoid(sig, grid) + S_T * 2 * gL / (np.pi * W)
    assert abs(area - S_T) / S_T < 2e-5


# -------------------------------------------------------- Beer-Lambert etc.
def _toy_mixture(x_co2=4e-4, x_n2o=3.4e-7, **kw):
    grid = np.arange(2290.0, 2310.0, 0.005)
    co2 = make_lines([2295.0, 2300.0, 2304.5], [3e-18, 1e-18, 2e-18], E=[50, 300, 900], M=2)
    n2o = make_lines([2299.0, 2302.0], [1e-18, 5e-19], E=[100, 400], M=4, mass=44.0)
    a1, a2 = absorption_coefficients(grid, co2, n2o, 255.0, 54020.0, x_co2, x_n2o, toy_Q,
                                        wing_cm1=10.0, **kw)
    return grid, a1, a2


def test_beer_lambert_consistency_and_bounds():
    grid, a1, a2 = _toy_mixture()
    I0 = np.linspace(1.0, 2.0, len(grid))
    tau, T, I = beer_lambert(a1 + a2, 1.0e4, I0)
    assert np.allclose(T, np.exp(-(a1 + a2) * 1.0e4))
    assert np.all((T >= 0) & (T <= 1))
    assert np.allclose(I, I0 * T)
    m = T > 0
    assert np.allclose(tau[m], -np.log(T[m]), rtol=1e-9, atol=1e-12)


def test_zero_absorber_gives_unit_transmission():
    grid, a1, a2 = _toy_mixture(x_co2=0.0, x_n2o=0.0)
    _, T, _ = beer_lambert(a1 + a2, 1e6, np.ones_like(grid))
    assert np.all(T == 1.0)


def test_low_abundance_and_short_path_limit():
    grid, a1, a2 = _toy_mixture(x_co2=1e-12, x_n2o=1e-12)
    tau, T, _ = beer_lambert(a1 + a2, 1.0, np.ones_like(grid))
    assert np.max(np.abs((1 - T) - tau)) < 1e-12           # 1 - exp(-tau) ~ tau


def test_optically_thick_limit():
    grid, a1, a2 = _toy_mixture(x_co2=0.2)
    _, T, _ = beer_lambert(a1 + a2, 1.0e6, np.ones_like(grid))
    assert T.min() < 1e-10


def test_mixture_additivity_and_linearity_in_mole_fraction():
    # with gamma_self == gamma_air the widths do not depend on x, so alpha is exactly linear in x
    g1, a1, _ = _toy_mixture(x_co2=2e-4)
    g2, a2, _ = _toy_mixture(x_co2=4e-4)
    assert np.allclose(2 * a1, a2, rtol=1e-12)
    # additivity: each gas computed ALONE must equal its component in the mixture,
    # so that alpha_total = alpha_CO2 + alpha_N2O with no cross-coupling in this model
    grid, c, n = _toy_mixture()
    _, c_alone, _ = _toy_mixture(x_n2o=0.0)
    _, _, n_alone = _toy_mixture(x_co2=0.0)
    assert np.allclose(c, c_alone, rtol=1e-12, atol=0)
    assert np.allclose(n, n_alone, rtol=1e-12, atol=0)
    tau_total = (c + n) * 1.0e4
    tau_sep = c * 1.0e4 + n * 1.0e4
    assert np.allclose(tau_total, tau_sep, rtol=1e-12, atol=0)


# --------------------------------------------------------- resolution tests
def test_resolution_convergence():
    rng = np.random.default_rng(1)
    nu = np.sort(rng.uniform(2295, 2305, 25))
    lines = make_lines(nu, rng.uniform(1e-19, 3e-18, 25), E=rng.uniform(0, 1500, 25))

    def integrated(step):
        grid = np.arange(2290.0, 2310.0, step)
        sig = species_cross_section(grid, lines, 255.0, 54020.0, 4e-4, toy_Q, wing_cm1=10.0)
        return trapezoid(sig, grid)
    ref = integrated(0.00125)
    err_fine = abs(integrated(0.005) - ref) / ref
    err_coarse = abs(integrated(0.1) - ref) / ref
    assert err_fine < 1e-4
    assert err_coarse > err_fine                           # error shrinks as resolution improves


# ----------------------------------------------------------- source and ISA
def test_planck_peak_follows_wien_law_in_wavenumber():
    T = 1500.0
    nu = np.arange(500.0, 8000.0, 0.5)
    peak = nu[np.argmax(planck_per_cm1(nu, T))]
    assert abs(peak - 2.8214393721 * T / C2_CM_K) < 1.0


def test_isa_troposphere_matches_tabulated_tropopause():
    T, P = isa_troposphere(11000.0)
    assert abs(T - 216.65) < 1e-9
    assert abs(P - 22632.1) / 22632.1 < 1e-3               # published table value
    T0, P0 = isa_troposphere(0.0)
    assert abs(T0 - 288.15) < 1e-12 and abs(P0 - 101325.0) < 1e-9


def test_dilution_warning():
    assert dilution_warnings(4e-4, 3e-7) == []
    assert len(dilution_warnings(0.2, 3e-7)) == 1

In [13]:
def run_tests():
    names = sorted(n for n, f in globals().items() if n.startswith("test_") and callable(f))
    failed = 0
    for n in names:
        try:
            globals()[n]()
            print("PASS", n)
        except Exception as exc:
            failed += 1
            print("FAIL", n, "->", repr(exc))
    print("\n%d/%d tests passed (synthetic data)" % (len(names) - failed, len(names)))

run_tests()

PASS test_beer_lambert_consistency_and_bounds
PASS test_c2_value
PASS test_dilution_warning
PASS test_doppler_width_scaling_and_value
PASS test_intensity_scaling_identity_and_formula
PASS test_isa_troposphere_matches_tabulated_tropopause
PASS test_line_strength_sum_rule
PASS test_loschmidt_number_density
PASS test_low_abundance_and_short_path_limit
PASS test_mixture_additivity_and_linearity_in_mole_fraction
PASS test_optically_thick_limit
PASS test_planck_peak_follows_wien_law_in_wavenumber
PASS test_pressure_and_temperature_dependence_of_lorentz_width
PASS test_resolution_convergence
PASS test_stimulated_emission_rayleigh_jeans_limit
PASS test_temperature_trends_differ_between_transitions
PASS test_voigt_limits_and_hwhm_convention
PASS test_voigt_normalisation
PASS test_zero_absorber_gives_unit_transmission

19/19 tests passed (synthetic data)


## 6. Load data and compute frames

Run `fetch_hitran()` once (or place `.par` files in `data/`), then load.

In [15]:
fetch_hitran()      # <- uncomment for the first run (needs your HITRAN account/API key set up for HAPI)

Using data

CO2_2155_2425
                     Lines parsed: 73495
N2O_2155_2425
                     Lines parsed: 43399

Data is fetched from http://hitran.org

http://hitran.org/lbl/api?iso_ids_list=7,8,9,10,11,12,13,14,121,15,120,122&numin=2155.0&numax=2425.0

BEGIN DOWNLOAD: CO2_2155_2425
  65536 bytes written to data/CO2_2155_2425.data
  65536 bytes written to data/CO2_2155_2425.data
  65536 bytes written to data/CO2_2155_2425.data
  65536 bytes written to data/CO2_2155_2425.data
  65536 bytes written to data/CO2_2155_2425.data
  65536 bytes written to data/CO2_2155_2425.data
  65536 bytes written to data/CO2_2155_2425.data
  65536 bytes written to data/CO2_2155_2425.data
  65536 bytes written to data/CO2_2155_2425.data
  65536 bytes written to data/CO2_2155_2425.data
  65536 bytes written to data/CO2_2155_2425.data
  65536 bytes written to data/CO2_2155_2425.data
  65536 bytes written to data/CO2_2155_2425.data
  65536 bytes written to data/CO2_2155_2425.data
  65536 bytes writt

In [16]:
CO2, rep_c, files_c = load_linelist("CO2", 2, cfg.DATA_DIR)
N2O, rep_n, files_n = load_linelist("N2O", 4, cfg.DATA_DIR)
print("CO2 lines: %d  | parameters replaced by data-derived estimates: %s" % (len(CO2.nu), rep_c))
print("N2O lines: %d  | parameters replaced by data-derived estimates: %s" % (len(N2O.nu), rep_n))
for msg in dilution_warnings(cfg.X_CO2, cfg.X_N2O):
    print(msg)
GRID = np.arange(cfg.NU_MIN, cfg.NU_MAX + 0.5 * cfg.NU_STEP, cfg.NU_STEP)
print("grid points:", len(GRID))

CO2 lines: 73495  | parameters replaced by data-derived estimates: {'gamma_air': 0, 'n_air': 0, 'gamma_self': 0}
N2O lines: 43399  | parameters replaced by data-derived estimates: {'gamma_air': 0, 'n_air': 0, 'gamma_self': 0}
grid points: 44001


In [17]:
# Conditions, frames, source spectrum, CSV export
# ----------------------------------------------------------------------------
def build_conditions(mode, n_frames, P_fixed_pa):
    """Return arrays (T, P) for each frame, sorted by ascending T."""
    if mode == "fixed-pressure":
        Ts = np.linspace(cfg.T_SWEEP[0], cfg.T_SWEEP[1], n_frames)
        Ps = np.full(n_frames, P_fixed_pa)
    else:                                   # isa-profile: T and P BOTH change
        h = np.linspace(0.0, 11000.0, n_frames)
        Ts, Ps = isa_troposphere(h)
    order = np.argsort(Ts)
    return Ts[order], Ps[order]


def compute_frames(grid, co2, n2o, Ts, Ps, x_co2, x_n2o, partition_fn):
    frames = []
    t0 = time.time()
    for i, (T, P) in enumerate(zip(Ts, Ps)):
        a_co2, a_n2o = absorption_coefficients(
            grid, co2, n2o, T, P, x_co2, x_n2o, partition_fn,
            wing_cm1=cfg.WING_CM1, s_min=cfg.S_MIN)
        frames.append(dict(T=T, P=P, a_co2=a_co2, a_n2o=a_n2o))
        print("  frame %2d/%d  T = %6.2f K  P = %8.1f Pa   (%.0f s elapsed)"
              % (i + 1, len(Ts), T, P, time.time() - t0))
    return frames


def pick_diagnostic_lines(lines, nu_min, nu_max):
    """Strongest line in the window and the highest-E'' line that is still
    reasonably strong (>= 1e-3 of the strongest): they respond to T differently."""
    inside = np.where((lines.nu >= nu_min) & (lines.nu <= nu_max))[0]
    k_strong = inside[np.argmax(lines.S[inside])]
    cand = inside[lines.S[inside] >= 1e-3 * lines.S[k_strong]]
    k_hot = cand[np.argmax(lines.E_low[cand])]
    return int(k_strong), int(k_hot)


def make_I0(grid, flat, t_source):
    if flat:
        return np.ones_like(grid), "arbitrary units (FLAT: I0 = 1)"
    return (planck_per_cm1(grid, t_source),
            "W m$^{-2}$ sr$^{-1}$ (cm$^{-1}$)$^{-1}$  (blackbody %.0f K)" % t_source)


# ----------------------------------------------------------------------------
# CSV export
# ----------------------------------------------------------------------------
def export_csv(path, grid, I0, frames, L_cm, x_co2, x_n2o, mode, notes=""):
    """Long-format CSV: one row per (frame, wavenumber). Comment lines start with '#'."""
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    cols = ["T_K", "P_Pa", "wavenumber_cm-1", "wavelength_um", "I0", "alpha_CO2_cm-1",
            "alpha_N2O_cm-1", "alpha_total_cm-1", "tau", "transmittance", "I_transmitted"]
    with open(path, "w") as fh:
        fh.write("# CO2+N2O transmission model; ILLUSTRATIVE simulation, not a validated product\n")
        fh.write("# x_CO2=%g x_N2O=%g L_cm=%g mode=%s %s\n" % (x_co2, x_n2o, L_cm, mode, notes))
        fh.write("# line parameters: HITRAN (see data/data_manifest.json); Voigt profile; Beer-Lambert\n")
        fh.write(",".join(cols) + "\n")
        for fr in frames:
            tot = fr["a_co2"] + fr["a_n2o"]
            tau, trans, I = beer_lambert(tot, L_cm, I0)
            block = np.column_stack([
                np.full_like(grid, fr["T"]), np.full_like(grid, fr["P"]), grid, 1.0e4 / grid, I0,
                fr["a_co2"], fr["a_n2o"], tot, tau, trans, I])
            np.savetxt(fh, block, delimiter=",", fmt="%.8g")
    print("Wrote", path)

## 7. Interactive animation

Panels: **A** incident spectrum (fixed) · **B** transmitted intensity · **C** transmittance (0–1) · **D** $\alpha_{CO_2}$, $\alpha_{N_2O}$ and their sum (additive)
· **E1** spectrally integrated optical depth vs $T$ · **E2** line strength $S(T)/S(296\,\mathrm K)$ for a strong and a high-$E''$ line of each gas.

Changing temperature, path length, source type or the selected wavenumber is instant. Changing pressure, composition or mode needs
**Recompute** (line widths change; it takes tens of seconds to a few minutes). The default sweeps $T$ at fixed $P$ (a what-if experiment);
mode `isa-profile` lets $T$ and $P$ both follow the standard troposphere. **Play** runs through the temperatures.

*If widgets do not render* (see the earlier ipywidgets/JupyterLab fix: `ipywidgets>=8`, `jupyterlab_widgets`, restart JupyterLab), call
`draw(idx, log10_L_in_m, "blackbody" or "flat", nu_sel)` after running the cell below; it works without widgets.

In [19]:
STATE = {}
P_LAYER = float(isa_troposphere(cfg.LAYER_H_M)[1])

def recompute(mode="fixed-pressure", P_pa=None, x_co2=None, x_n2o=None):
    # (Re)compute one absorption-coefficient frame per temperature; store everything in STATE.
    P_pa = P_LAYER if P_pa is None else P_pa
    x_co2 = cfg.X_CO2 if x_co2 is None else x_co2
    x_n2o = cfg.X_N2O if x_n2o is None else x_n2o
    for msg in dilution_warnings(x_co2, x_n2o):
        print(msg)
    Ts, Ps = build_conditions(mode, cfg.N_FRAMES, P_pa)
    print("Computing %d frames on %d grid points ..." % (len(Ts), len(GRID)))
    frames = compute_frames(GRID, CO2, N2O, Ts, Ps, x_co2, x_n2o, hapi_partition_fn)
    allpos = np.concatenate([f["a_co2"] + f["a_n2o"] for f in frames])
    allpos = allpos[allpos > 0]
    lo = max(np.percentile(allpos, 1.0) / 10.0, 1e-14) if len(allpos) else 1e-14
    hi = allpos.max() * 3 if len(allpos) else 1.0
    diag = []
    for lines in (CO2, N2O):
        for k in pick_diagnostic_lines(lines, GRID[0], GRID[-1]):
            S_T = line_strength_vs_T(lines, k, Ts, hapi_partition_fn)
            S296 = line_strength_vs_T(lines, k, [T_REF], hapi_partition_fn)[0]
            diag.append(("%s %.2f cm-1, E''=%.0f" % (lines.name, lines.nu[k], lines.E_low[k]), S_T / S296))
    STATE.update(frames=frames, Ts=Ts, mode=mode, P_pa=P_pa, x_co2=x_co2, x_n2o=x_n2o, ylimD=(lo, hi), diag=diag)
    print("done.")

def default_nu_sel():
    inside = (N2O.nu >= GRID[0]) & (N2O.nu <= GRID[-1])
    return float(N2O.nu[inside][np.argmax(N2O.S[inside])])      # strongest N2O line

def draw(idx, logL, src, nu_sel):
    # Draw all panels for temperature frame `idx`, path length 10**logL metres, source 'blackbody'/'flat'.
    frames, Ts = STATE["frames"], STATE["Ts"]
    fr = frames[idx]
    L_cm = 10 ** logL * 100.0
    I0, I0_unit = make_I0(GRID, src == "flat", cfg.T_SOURCE_K)
    tot = fr["a_co2"] + fr["a_n2o"]
    tau, trans, I = beer_lambert(tot, L_cm, I0)
    i_sel = int(np.argmin(np.abs(GRID - nu_sel)))
    W = equivalent_width(GRID, trans)
    int_co2 = np.array([integrated_tau(GRID, f["a_co2"], L_cm) for f in frames])
    int_n2o = np.array([integrated_tau(GRID, f["a_n2o"], L_cm) for f in frames])

    fig, ax = plt.subplots(3, 2, figsize=(14, 10.5))
    fig.subplots_adjust(top=0.86, bottom=0.06, hspace=0.55, wspace=0.22)
    fig.suptitle(
        "CO2 + N2O slab, %.0f-%.0f cm-1 (%.2f-%.2f um)   [ILLUSTRATIVE settings, HITRAN line data]\n"
        "T = %.2f K   P = %.0f Pa (%s)   x_CO2 = %.2f ppm   x_N2O = %.1f ppb   L = %.3g m\n"
        "selected %.3f cm-1: tau = %.3e, T = %.3e     equivalent width W = %.3f cm-1 (flat-I0 measure, not absorbed energy)"
        % (GRID[0], GRID[-1], 1e4 / GRID[-1], 1e4 / GRID[0], fr["T"], fr["P"],
           "P follows ISA" if STATE["mode"] == "isa-profile" else "P fixed",
           STATE["x_co2"] * 1e6, STATE["x_n2o"] * 1e9, L_cm / 100.0, GRID[i_sel], tau[i_sel], trans[i_sel], W),
        fontsize=9, family="monospace")

    ax[0, 0].plot(GRID, I0, "k"); ax[0, 0].set_title("A. Incident spectrum I0 (fixed)", fontsize=10)
    ax[0, 0].set_ylabel(I0_unit, fontsize=8)
    ax[0, 1].plot(GRID, I0, color="0.7", lw=1, label="I0"); ax[0, 1].plot(GRID, I, "C3", lw=0.8, label="I transmitted")
    ax[0, 1].set_ylim(0, 1.05 * I0.max()); ax[0, 1].legend(fontsize=7, loc="lower right")
    ax[0, 1].set_title("B. Transmitted spectral intensity  I = I0 exp(-tau)", fontsize=10)
    ax[1, 0].plot(GRID, trans, "C0", lw=0.8); ax[1, 0].set_ylim(0, 1.02)
    ax[1, 0].set_title("C. Transmittance  I/I0 = exp(-tau)  (dimensionless)", fontsize=10)
    lo, hi = STATE["ylimD"]
    ax[1, 1].semilogy(GRID, np.maximum(fr["a_co2"], lo), "C1", lw=0.8, label="alpha_CO2")
    ax[1, 1].semilogy(GRID, np.maximum(fr["a_n2o"], lo), "C2", lw=0.8, label="alpha_N2O")
    ax[1, 1].semilogy(GRID, np.maximum(tot, lo), "k", lw=0.8, label="alpha_total = alpha_CO2 + alpha_N2O (additive)")
    ax[1, 1].set_ylim(lo, hi); ax[1, 1].legend(fontsize=7, loc="upper left")
    ax[1, 1].set_ylabel("absorption coeff. (cm-1)", fontsize=8)
    ax[1, 1].set_title("D. Individual and combined absorption coefficients", fontsize=10)
    for a in (ax[0, 0], ax[0, 1], ax[1, 0], ax[1, 1]):
        a.set_xlabel("wavenumber (cm-1)", fontsize=8); a.tick_params(labelsize=8)
    for a in (ax[1, 0], ax[1, 1]):
        a.axvline(GRID[i_sel], color="g", ls=":", lw=1)
    ax[2, 0].semilogy(Ts, np.maximum(int_co2, 1e-300), "o-", ms=3, color="C1", label="CO2")
    ax[2, 0].semilogy(Ts, np.maximum(int_n2o, 1e-300), "o-", ms=3, color="C2", label="N2O")
    ax[2, 0].semilogy(Ts, np.maximum(int_co2 + int_n2o, 1e-300), "o-", ms=3, color="k", label="total")
    pos = np.concatenate([int_co2, int_n2o]); pos = pos[pos > 0]
    if len(pos):
        ax[2, 0].set_ylim(pos.min() / 3, (int_co2 + int_n2o).max() * 3)
    ax[2, 0].axvline(fr["T"], color="r"); ax[2, 0].legend(fontsize=7)
    ax[2, 0].set_xlabel("gas temperature T (K)", fontsize=8); ax[2, 0].set_ylabel("integral of tau d(nu)  (cm-1)", fontsize=8)
    ax[2, 0].set_title("E1. Spectrally integrated optical depth vs T", fontsize=10)
    for name, ratio in STATE["diag"]:
        ax[2, 1].semilogy(Ts, ratio, "o-", ms=3, label=name)
    ax[2, 1].axvline(fr["T"], color="r"); ax[2, 1].legend(fontsize=6)
    ax[2, 1].set_xlabel("gas temperature T (K)", fontsize=8); ax[2, 1].set_ylabel("S(T) / S(296 K)", fontsize=8)
    ax[2, 1].set_title("E2. Line strength vs T: different lines, different trends", fontsize=10)
    for a in (ax[2, 0], ax[2, 1]):
        a.tick_params(labelsize=8)
    plt.show()

recompute()                                   # default: 5 km layer, fixed pressure
STATE["nu_sel"] = default_nu_sel()

Computing 13 frames on 44001 grid points ...
  frame  1/13  T = 216.65 K  P =  54019.9 Pa   (4 s elapsed)
  frame  2/13  T = 222.61 K  P =  54019.9 Pa   (9 s elapsed)
  frame  3/13  T = 228.57 K  P =  54019.9 Pa   (13 s elapsed)
  frame  4/13  T = 234.53 K  P =  54019.9 Pa   (17 s elapsed)
  frame  5/13  T = 240.48 K  P =  54019.9 Pa   (21 s elapsed)
  frame  6/13  T = 246.44 K  P =  54019.9 Pa   (25 s elapsed)
  frame  7/13  T = 252.40 K  P =  54019.9 Pa   (30 s elapsed)
  frame  8/13  T = 258.36 K  P =  54019.9 Pa   (34 s elapsed)
  frame  9/13  T = 264.32 K  P =  54019.9 Pa   (38 s elapsed)
  frame 10/13  T = 270.27 K  P =  54019.9 Pa   (42 s elapsed)
  frame 11/13  T = 276.23 K  P =  54019.9 Pa   (47 s elapsed)
  frame 12/13  T = 282.19 K  P =  54019.9 Pa   (51 s elapsed)
  frame 13/13  T = 288.15 K  P =  54019.9 Pa   (55 s elapsed)
done.


In [20]:
# ---- widgets ----
IDX0 = int(np.argmin(np.abs(STATE["Ts"] - isa_troposphere(cfg.LAYER_H_M)[0])))
def _opts():
    return [("%.2f K" % T, i) for i, T in enumerate(STATE["Ts"])]

play = widgets.Play(min=0, max=len(STATE["Ts"]) - 1, step=1, interval=900, value=IDX0, description="Play")
T_slider = widgets.SelectionSlider(options=_opts(), index=IDX0, description="Temperature",
                                   continuous_update=False, layout=widgets.Layout(width="520px"),
                                   style={"description_width": "initial"})
L_slider = widgets.FloatSlider(value=np.log10(cfg.PATH_LENGTH_M), min=-1.0, max=4.0, step=0.1, continuous_update=False,
                               description="log10 L (m)", layout=widgets.Layout(width="420px"),
                               style={"description_width": "initial"})
src_dd = widgets.Dropdown(options=[("blackbody %.0f K" % cfg.T_SOURCE_K, "blackbody"), ("flat (I0 = 1)", "flat")],
                          value="blackbody", description="Incident I0")
nu_box = widgets.FloatText(value=STATE["nu_sel"], description="selected nu (cm-1)", style={"description_width": "initial"})
mode_dd = widgets.Dropdown(options=["fixed-pressure", "isa-profile"], value="fixed-pressure", description="mode")
P_box = widgets.FloatText(value=round(P_LAYER, 1), description="P (Pa)")
xc_box = widgets.FloatText(value=cfg.X_CO2 * 1e6, description="CO2 (ppm)")
xn_box = widgets.FloatText(value=cfg.X_N2O * 1e9, description="N2O (ppb)")
btn_recompute = widgets.Button(description="Recompute (P / composition / mode)", button_style="warning",
                               layout=widgets.Layout(width="300px"))
btn_csv = widgets.Button(description="Export CSV (current T, L)", layout=widgets.Layout(width="220px"))
out = widgets.Output()
status = widgets.Output()
widgets.link((play, "value"), (T_slider, "index"))

def redraw(*_):
    with out:
        clear_output(wait=True)
        draw(T_slider.index, L_slider.value, src_dd.value, nu_box.value)

def on_recompute(_):
    with status:
        clear_output()
        play.max = 0
        recompute(mode_dd.value, P_box.value, xc_box.value * 1e-6, xn_box.value * 1e-9)
        T_slider.options = _opts()
        play.max = len(STATE["Ts"]) - 1
        T_slider.index = int(np.argmin(np.abs(STATE["Ts"] - isa_troposphere(cfg.LAYER_H_M)[0])))
    redraw()

def on_csv(_):
    fr = STATE["frames"][T_slider.index]
    L_cm = 10 ** L_slider.value * 100.0
    I0, _ = make_I0(GRID, src_dd.value == "flat", cfg.T_SOURCE_K)
    path = os.path.join("outputs", "spectrum_T%.1fK_L%.0fm.csv" % (fr["T"], L_cm / 100.0))
    with status:
        export_csv(path, GRID, I0, [fr], L_cm, STATE["x_co2"], STATE["x_n2o"], STATE["mode"])

T_slider.observe(redraw, names="index")
for w in (L_slider, src_dd, nu_box):
    w.observe(redraw, names="value")
btn_recompute.on_click(on_recompute)
btn_csv.on_click(on_csv)

display(widgets.VBox([
    widgets.HBox([play, T_slider]),
    widgets.HBox([L_slider, src_dd, nu_box]),
    widgets.HBox([mode_dd, P_box, xc_box, xn_box]),
    widgets.HBox([btn_recompute, btn_csv]),
    status, out]))
redraw()

## 8. Export

The **Export CSV** button writes the current frame. To write *all* temperature frames in one long CSV:

In [22]:
I0_csv, _ = make_I0(GRID, False, cfg.T_SOURCE_K)
export_csv("outputs/all_frames.csv", GRID, I0_csv, STATE["frames"], cfg.PATH_LENGTH_M * 100.0,
           STATE["x_co2"], STATE["x_n2o"], STATE["mode"])

Wrote outputs/all_frames.csv


## 9. Checks you should run on real data

These close the gaps listed in the status note. Nothing here has been run on real HITRAN data by me.

1. **Interval check** – are both gases visible and distinguishable? Inspect the strongest lines and band structure.
2. **Sensitivity** to the line-wing cut-off, the weak-line threshold and the grid step (the N₂O region sits in the far wing of the CO₂ band).
3. **Cross-check against HAPI's own Voigt routine** for the same lines.
4. **Real-data sanity**: with a long path, CO₂ should be black near 2349 cm⁻¹; with a very short path $\mathcal T\to1$ everywhere.

In [24]:
def strongest_lines(lines, n=8, lo=None, hi=None):
    lo = GRID[0] if lo is None else lo; hi = GRID[-1] if hi is None else hi
    sel = np.where((lines.nu >= lo) & (lines.nu <= hi))[0]
    top = sel[np.argsort(lines.S[sel])[::-1][:n]]
    print(lines.name, ": %d lines in %.0f-%.0f cm-1" % (len(sel), lo, hi))
    for k in top:
        print("  nu = %9.4f cm-1   S = %.3e   E'' = %8.2f   gamma_air = %.4f   iso %d"
              % (lines.nu[k], lines.S[k], lines.E_low[k], lines.gamma_air[k], lines.iso[k]))

strongest_lines(CO2); strongest_lines(N2O)

CO2 : 68162 lines in 2180-2400 cm-1
  nu = 2361.4659 cm-1   S = 3.542e-18   E'' =   106.13   gamma_air = 0.0742   iso 1
  nu = 2360.1090 cm-1   S = 3.515e-18   E'' =    81.94   gamma_air = 0.0755   iso 1
  nu = 2362.7980 cm-1   S = 3.466e-18   E'' =   133.44   gamma_air = 0.0731   iso 1
  nu = 2358.7272 cm-1   S = 3.375e-18   E'' =    60.87   gamma_air = 0.0770   iso 1
  nu = 2335.9194 cm-1   S = 3.330e-18   E'' =   106.13   gamma_air = 0.0742   iso 1
  nu = 2364.1053 cm-1   S = 3.304e-18   E'' =   163.87   gamma_air = 0.0722   iso 1
  nu = 2334.1563 cm-1   S = 3.281e-18   E'' =   133.44   gamma_air = 0.0731   iso 1
  nu = 2337.6580 cm-1   S = 3.278e-18   E'' =    81.94   gamma_air = 0.0755   iso 1
N2O : 29669 lines in 2180-2400 cm-1
  nu = 2236.2234 cm-1   S = 1.002e-18   E'' =   100.55   gamma_air = 0.0777   iso 1
  nu = 2235.4964 cm-1   S = 9.990e-19   E'' =    87.98   gamma_air = 0.0783   iso 1
  nu = 2236.9436 cm-1   S = 9.984e-19   E'' =   113.96   gamma_air = 0.0770   iso 1
  nu

In [25]:
def sensitivity(T=None, P_pa=None, L_m=None, window=(2200.0, 2260.0)):
    # How much do the chosen numerical approximations matter inside `window`?
    T = float(isa_troposphere(cfg.LAYER_H_M)[0]) if T is None else T
    P_pa = P_LAYER if P_pa is None else P_pa
    L_cm = (cfg.PATH_LENGTH_M if L_m is None else L_m) * 100.0
    base = dict(step=cfg.NU_STEP, wing=cfg.WING_CM1, smin=cfg.S_MIN)
    variants = [("baseline", {}),
                ("wing 5 cm-1", dict(wing=5.0)), ("wing 25 cm-1", dict(wing=25.0)), ("wing 50 cm-1", dict(wing=50.0)),
                ("S_min x100", dict(smin=cfg.S_MIN * 100)), ("S_min /100", dict(smin=cfg.S_MIN / 100)),
                ("step x2", dict(step=cfg.NU_STEP * 2)), ("step /2", dict(step=cfg.NU_STEP / 2))]
    print("window %.0f-%.0f cm-1, T = %.2f K, P = %.0f Pa, L = %.3g m" % (window + (T, P_pa, L_cm / 100)))
    print("%-14s %14s %14s %14s" % ("variant", "int tau dnu", "mean T", "rel. change"))
    ref = None
    for name, ch in variants:
        p = dict(base, **ch)
        g = np.arange(window[0], window[1], p["step"])
        a1, a2 = absorption_coefficients(g, CO2, N2O, T, P_pa, STATE["x_co2"], STATE["x_n2o"], hapi_partition_fn,
                                         wing_cm1=p["wing"], s_min=p["smin"])
        itau = integrated_tau(g, a1 + a2, L_cm)
        mT = trapezoid(np.exp(-(a1 + a2) * L_cm), g) / (g[-1] - g[0])
        ref = itau if ref is None else ref
        print("%-14s %14.6e %14.6e %14.3e" % (name, itau, mT, (itau - ref) / ref))

# sensitivity()      # takes a few minutes; shrink `window` for speed

In [26]:
def check_against_hapi(name="N2O", nu_a=2220.0, nu_b=2230.0, T_list=(296.0, 255.65), P_atm=0.533, x_self=3.4e-7):
    # Compare our cross section with HAPI's absorptionCoefficient_Voigt for the same lines (needs data/ files).
    # Cores should agree closely at 296 K; far wings and T != 296 K can differ because of cut-off and exponent conventions.
    import hapi
    M = dict(CO2=2, N2O=4)[name]
    hapi.db_begin(cfg.DATA_DIR)
    table = [os.path.splitext(os.path.basename(p))[0]
             for p in glob.glob(os.path.join(cfg.DATA_DIR, name + "*.data")) + glob.glob(os.path.join(cfg.DATA_DIR, name + "*.par"))][0]
    lines = CO2 if name == "CO2" else N2O
    g = np.arange(nu_a, nu_b, cfg.NU_STEP)
    for T in T_list:
        mine = species_cross_section(g, lines, T, P_atm * ATM_PA, x_self, hapi_partition_fn, wing_cm1=cfg.WING_CM1, s_min=0.0)
        nu_h, sig_h = hapi.absorptionCoefficient_Voigt(SourceTables=[table], Environment={"T": T, "p": P_atm},
                       WavenumberRange=(nu_a, nu_b), WavenumberStep=cfg.NU_STEP, HITRAN_units=True,
                       Diluent={"air": 1.0 - x_self, "self": x_self})
        ref = np.interp(g, nu_h, sig_h)
        core = mine > 0.2 * mine.max()
        rel = np.abs(mine[core] - ref[core]) / ref[core]
        print("%s T=%.2f K: core points %d, max |rel diff| = %.3e, median = %.3e" % (name, T, core.sum(), rel.max(), np.median(rel)))

# check_against_hapi()   # check HAPI's signature first: hapi.getHelp(hapi.absorptionCoefficient_Voigt)

## 10. References (what each source supports)

* HITRAN, *Definitions and units*, https://hitran.org/docs/definitions-and-units/ — units of $S$, $\gamma$ (HWHM at 296 K and 1 atm), $n_{air}$, and the temperature scaling of $S$.
* Kochanov et al., "HITRAN Application Programming Interface (HAPI)…", JQSRT **177**, 15–30 (2016), doi:10.1016/j.jqsrt.2016.03.005; https://hitran.org/hapi/ — HAPI; key requirement and daily request limit.
* Gordon et al., "The HITRAN2024 molecular spectroscopic database", JQSRT **353**, 109807 (2026), doi:10.1016/j.jqsrt.2026.109807 — database edition (as listed on the HAPI page).
* Gamache et al., "Partition sums for molecules and their isotopologues for HITRAN2024", JQSRT **345**, 109568 (2025), doi:10.1016/j.jqsrt.2025.109568 — TIPS partition sums used by HAPI.
* Hargreaves et al., "Spectroscopic line parameters of NO, NO₂, and N₂O for the HITEMP database", JQSRT **232**, 35–53 (2019) — high-temperature N₂O (¹⁴N₂¹⁶O only).
* Hashemi et al., HITRAN2020 air/self-broadening updates for N₂O and CO, and "Revising the line-shape parameters for air- and self-broadened CO₂ lines toward a sub-percent accuracy level" (2020) — origin of the broadening parameters (journal details not verified here).
* *U.S. Standard Atmosphere, 1976*, U.S. Government Printing Office — $T_0=288.15$ K, $P_0=101325$ Pa, 6.5 K/km lapse rate, tropopause 11 km (216.65 K, 22 632 Pa). NASA NTRS copy: https://ntrs.nasa.gov/api/citations/19770009539/downloads/19770009539.pdf
* NOAA GML, CO₂ trends (423.03 ppm, Sep 2025): https://gml.noaa.gov/ccgg/trends/global.html · N₂O trends (339.76 ppb, Mar 2026): https://www.gml.noaa.gov/ccgg/trends_n2o
* Doppler width, the Voigt profile via the Faddeeva function (SciPy `wofz`), Beer–Lambert law, and the Planck function in wavenumber form are standard results used without a verified equation number. **No equation or page numbers are claimed.**